# Predizione

In [ ]:
import torch


import sys
from pathlib import Path

# aggiungo src/ModelloFinale al path
sys.path.append(str(Path("..") / "ModelloFinale"))

from Modello import *
from Pulisci_testo import *
from Tokenize import *




# =========================
# 2) CARICO token2id dal vocabolario salvato
# =========================
DIR_VOC = Path.cwd().parent / "Datasets_puliti" / "vocabolario.pkl"   
WEIGHTS = Path.cwd().parent / "Code" / "best_model.pt"
token2id = carica_token2id(DIR_VOC)

MAX_LEN = 64
PAD_ID  = token2id["<PAD>"]
UNK_ID  = token2id["<UNK>"]

# =========================
# 3) DEVICE + MODELLO + PESI
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = IronyEndToEndModel(
    dim_vocabolario=len(token2id),
    dim_wordVector=128,
    dim_frase=MAX_LEN,
    num_heads=4
).to(device)

model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()

# =========================
# 4) FUNZIONE DI PREDIZIONE
# =========================
def predict_sentence(text: str):
    # --- preprocessing ---
    text_clean = pulisci_testo(text)
    tokens = tokenize(text_clean)

    # --- token -> id ---
    token_ids = [token2id.get(tok, UNK_ID) for tok in tokens]

    # --- padding/truncate + mask ---
    if len(token_ids) >= MAX_LEN:
        input_ids = token_ids[:MAX_LEN]
        attention_mask = [1] * MAX_LEN
    else:
        pad_len = MAX_LEN - len(token_ids)
        input_ids = token_ids + [PAD_ID] * pad_len
        attention_mask = [1] * len(token_ids) + [0] * pad_len

    # --- tensori (batch=1) ---
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)          # [1,64]
    attention_mask = torch.tensor([attention_mask], dtype=torch.long, device=device) # [1,64]

    # --- forward ---
    with torch.no_grad():
        logit = model(input_ids, attention_mask)          # [1]
        prob = torch.sigmoid(logit).item()                # float
        pred = int(prob >= 0.5)

    return pred, prob



C:\Users\marco\AppData\Local\Temp\ipykernel_24776\324441683.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(WEIGHTS, map_location=devic

In [3]:
Prova = "sei arrivato presto oggi "

pred, prob = predict_sentence(Prova)
print("Predizione:", pred, "Probabilità:", round(prob, 3))

Predizione: 0 Probabilità: 0.022
